# Sentiment Volume Forecast

Prediksi jumlah postingan / akun per hari untuk tiap sentimen dari data export media sosial, lengkap dengan uji ketepatan ke data lama dan grafik interaktif.

<a href="https://colab.research.google.com/github/aldo02032004/sentiment-volume-forecast/blob/main/notebooks/sentiment_volume_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Panduan pengaturan ideal
Slider sudah otomatis dibatasi ke rentang aman.

| Panjang data | Prediksi 1 – 3 hari | Prediksi 4 – 7 hari | Catatan |
|---|---|---|---|
| < 21 hari | – | – | data terlalu pendek, belum bisa diuji |
| 21 – 45 hari | belajar **7** | belajar **7** | skor ketepatan masih kasar kalau uji < 14 kali |
| 46 – 89 hari | belajar **7 – 14** | belajar **14** | |
| ≥ 90 hari | belajar **28 – 42** | belajar **28 – 42** | Regresi pintar paling unggul |

**Jumlah uji** = panjang data − hari dipelajari − hari diprediksi + 1 *(minimal 14, maksimal 60 hari terakhir)*


# 1. Install

In [ ]:
!pip install -q "great[viz] @ git+https://github.com/azmkto/GREAT-Tools.git"

# 2. Library

In [ ]:
import io, re, time, zipfile          # io: baca file dari memori ; re: ambil ID dari link drive
                                      # time: jeda sebelum unduh ulang ; zipfile: buka file zip berisi Excel

import numpy as np                        # hitungan angka
import pandas as pd                       # tabel data
import requests                           # unduh file dari Google Drive
import ipywidgets as w                    # slider, dropdown, tombol
import matplotlib.pyplot as plt           # gambar grafik
from matplotlib.lines import Line2D       # contoh garis putus-putus di keterangan grafik
from matplotlib.patches import Patch      # contoh area bayangan di keterangan grafik
from IPython.display import display       # tampilin tabel & interaktif
from sklearn.linear_model import LinearRegression, Ridge   # scikit-learn: LinearRegression = tren linear; Ridge = regresi dengan penahan

from great import sent_class, sent_colors, validate_export   # library GREAT: urutan & warna sentimen, format file export
from great.viz import prep                                   # ngerapiin data export (tanggal, sentimen, dll.)
from great.viz.overview import _panel_sentiment_trend        # grafik tren sentimen yang sama dengan monthly report
from great.viz.style import apply_style                      # grafik style

print("Library siap")

# 3. Pengaturan

In [ ]:
SOURCES = [   # link Google Drive (zip, excel,sheets)
    "https://drive.google.com/file/d/YOUR_FILE_ID/view?usp=sharing",
]
OBJECT_NAME = "Someone"   # nama objek penelitian

METHODS = ["Hari terakhir", "Rata-rata", "Minggu lalu", "Tren linear", "Regresi pintar"]
TEST_DAYS = 60        # uji ketepatan ke 60 hari terakhir
MIN_TESTS = 14        # minimal jumlah uji supaya skor bisa dipercaya
BAND = 0.80           # lebar area bayangan: 80% angka asli masuk area ini
RIDGE_ALPHA = 10      # seberapa kuat Regresi pintar menahan efek per hari

# 4. Load data

In [ ]:
def download(link, label, tries=3):
    file_id = (re.search(r"/d/([\w-]+)", link) or re.search(r"[?&]id=([\w-]+)", link)).group(1)
    urls = [f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t",  # file Excel / zip
            f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"]                    # Google Sheets
    for attempt in range(1, tries + 1):
        for url in urls:
            try:
                resp = requests.get(url, timeout=300)
            except requests.RequestException:
                continue
            if resp.ok and resp.content[:2] in (b"PK", b"\xd0\xcf"):   # re-check
                return resp.content
        time.sleep(3 * attempt)   # jeda sebelum coba lagi
    raise RuntimeError(f"{label} gagal diunduh. Cek akses 'Anyone with the link' & koneksi internet.")

def load_link(link, label):
    content = download(link, label)
    if content[:2] == b"\xd0\xcf":
        return [pd.read_excel(io.BytesIO(content), header=1)]
    with zipfile.ZipFile(io.BytesIO(content)) as z:
        names = z.namelist()
        if "xl/workbook.xml" in names:
            return [pd.read_excel(io.BytesIO(content), header=1)]
        excel = [n for n in names if n.lower().endswith((".xlsx", ".xls"))
                 and not n.startswith("__MACOSX/") and "/~$" not in "/" + n]
        print(f"{label}: zip isi {len(excel)} file Excel")
        return [pd.read_excel(io.BytesIO(z.read(n)), header=1) for n in excel]


df = pd.concat([validate_export(t) for i, s in enumerate(SOURCES, 1) for t in load_link(s, f"file_{i}")],
               ignore_index=True)
df["Date"] = pd.to_datetime(df["Date"], format="ISO8601")
before = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "No"]).reset_index(drop=True)   # buang postingan double dari file yang tanggalnya tumpang tindih
df1, start_date, end_date = prep.prepare_data(df)
print(f"data yang digunakan: {len(df1):,} baris ({before - len(df):,} terindikasi redundan di cut) | {start_date} - {end_date}")

# 5. Hitung per hari

In [ ]:
group = df1.groupby([pd.Grouper(key="Date", freq="D"), "Sentiment"])
daily = {
    "Postingan": group.size(),               # jumlah postingan per hari
    "Akun": group["Author"].nunique(),       # jumlah akun berbeda per hari
}
daily = {k: v.unstack(fill_value=0).reindex(columns=sent_class, fill_value=0).asfreq("D", fill_value=0).astype(float)
         for k, v in daily.items()}

n_days = len(daily["Postingan"])
if n_days < 7 + MIN_TESTS:                   # minimal 7 hari dipelajari + MIN_TESTS kali uji
    raise ValueError(f"Data cuma {n_days} hari, minimal {7 + MIN_TESTS} hari")
print(f"Total data: {n_days} hari")
for k, v in daily.items():
    print(f"\n{k} 10 hari terakhir:")
    display(v.tail(10).astype(int))

# 6. Metode prediksi

In [ ]:
# fitur kalender per hari: [akhir pekan, Senin..Minggu] isinya 1/0
def calendar(dates):
    dow = dates.dayofweek.to_numpy()
    return np.column_stack([dow >= 5, dow[:, None] == np.arange(7)]).astype(float)


# prediksi `horizon` hari ke depan, belajar dari `lookback` hari terakhir
def forecast(y, method, lookback, horizon):
    hist = y.iloc[-lookback:]
    future = pd.date_range(y.index[-1] + pd.Timedelta(days=1), periods=horizon, freq="D")

    if method == "Hari terakhir":         # besok = sama dengan hari terakhir
        pred = np.tile(hist.iloc[-1], (horizon, 1))
    elif method == "Rata-rata":           # besok = rata-rata hari yang dipelajari
        pred = np.tile(hist.mean(), (horizon, 1))
    elif method == "Minggu lalu":         # besok = hari yang sama minggu lalu
        pred = y.iloc[-7:].to_numpy()[np.arange(horizon) % 7]
    elif method == "Tren linear":         # tarik garis lurus dari hari yang dipelajari
        model = LinearRegression().fit(np.arange(lookback)[:, None], hist)
        pred = model.predict(np.arange(lookback, lookback + horizon)[:, None])
    elif method == "Regresi pintar":
        # log(1 + hari ini) ditebak dari log(1 + kemarin) + efek hari; tebakan besok dipakai buat lusa, dst.
        z, cal, pred = np.log1p(hist.to_numpy()), calendar(hist.index), []
        for j in range(z.shape[1]):   # satu model per sentimen
            model = Ridge(alpha=RIDGE_ALPHA).fit(np.column_stack([z[:-1, j], cal[1:]]), z[1:, j])
            last, out = z[-1, j], []
            for c in calendar(future):
                last = model.intercept_ + model.coef_ @ np.r_[last, c]
                out.append(last)
            pred.append(np.expm1(out))
        pred = np.column_stack(pred)
    else:
        raise ValueError(f"Metode tidak dikenal: {method}")
    return pd.DataFrame(np.clip(pred, 0, None), index=future, columns=y.columns)   # jumlah nggak mungkin minus


# uji ke data lama: pura-pura berdiri di hari lama, prediksi, lalu bandingin sama angka aslinya
def backtest(y, methods, lookback, horizon, test_days):
    ends = range(max(lookback, 7, len(y) - horizon + 1 - test_days), len(y) - horizon + 1)
    errors = {m: [] for m in methods}   # angka asli - prediksi, per uji
    for end in ends:
        actual = y.iloc[end:end + horizon].to_numpy()
        for m in methods:
            errors[m].append(actual - forecast(y.iloc[:end], m, lookback, horizon).to_numpy())
    total = sum(y.iloc[end:end + horizon].to_numpy().sum() for end in ends)
    accuracy = {m: 1 - np.abs(e).sum() / total for m, e in errors.items()}   # 100% - rata-rata meleset
    return accuracy, {m: np.array(e) for m, e in errors.items()}, len(ends)


print("Metode siap:", ", ".join(METHODS))

# 7. Visual

In [ ]:
# pengaturan tampilan grafik
FIGSIZE = (30, 8)          # (lebar, tinggi) grafik
DPI = 100                  # ketajaman grafik: makin besar makin tajam, tapi slider makin lambat
COLOR_LEARNED = "orange"   # warna kotak hari yang dipelajari
COLOR_LEGEND = "grey"      # warna contoh garis & bayangan di keterangan grafik
AUTO = "Otomatis (paling akurat)"


# rentang aman slider, ngikutin panjang data
def max_horizon(n):
    # prediksi paling jauh: 1/4 panjang data, maksimal 14 hari, dan masih nyisain MIN_TESTS kali uji
    return max(1, min(14, n // 4, n - 6 - MIN_TESTS))


def lookback_range(n, horizon):
    low = max(2, min(7, n - horizon))   # minimal 7 hari (1 minggu penuh)
    return low, max(low, n - horizon + 1 - MIN_TESTS)   # maksimal: masih nyisain MIN_TESTS kali uji


# hasil uji disimpan, jadi kalau slider digeser balik ke posisi lama nggak ngitung ulang
_cache = {}


def run(basis, horizon, lookback, method):
    key = (basis, horizon, lookback)
    if key not in _cache:
        # garis tren yang ditarik jauh ke depan hampir selalu meleset, jadi cuma dipakai buat prediksi <= 4 hari
        methods = [m for m in METHODS if not (m == "Tren linear" and horizon > 4)]
        _cache[key] = backtest(daily[basis], methods, lookback, horizon, TEST_DAYS)
    accuracy, errors, n_tests = _cache[key]

    best = method if method in accuracy else max(accuracy, key=accuracy.get)   # Otomatis -> metode paling akurat
    pred = forecast(daily[basis], best, lookback, horizon)
    # area bayangan = prediksi + sebaran meleset di uji data lama (per hari ke depan)
    low = (pred + np.quantile(errors[best], (1 - BAND) / 2, axis=0)).clip(lower=0)
    high = (pred + np.quantile(errors[best], (1 + BAND) / 2, axis=0)).clip(lower=0)
    return best, accuracy, n_tests, pred, low, high


# pilihan interaktif, batas slider otomatis di rentang aman
apply_style(dpi=DPI)
style = {"description_width": "140px"}
basis_w = w.ToggleButtons(options=list(daily), description="Yang dihitung", style=style)
method_w = w.Dropdown(options=[AUTO] + METHODS, value=AUTO, description="Metode", style=style)
horizon_w = w.IntSlider(value=min(3, max_horizon(n_days)), min=1, max=max_horizon(n_days),
                        description="Hari diprediksi", style=style)
low_lb, high_lb = lookback_range(n_days, horizon_w.value)
lookback_w = w.IntSlider(value=min(7, high_lb), min=low_lb, max=high_lb, description="Hari dipelajari", style=style)


# kalau "hari diprediksi" digeser, rentang "hari dipelajari" ikut nyesuaiin
def update_lookback(change):
    low_lb, high_lb = lookback_range(n_days, horizon_w.value)
    if low_lb > lookback_w.max:
        lookback_w.max, lookback_w.min = high_lb, low_lb
    else:
        lookback_w.min, lookback_w.max = low_lb, high_lb


horizon_w.observe(update_lookback, names="value")


def draw(basis, horizon, lookback, method):
    best, accuracy, n_tests, pred, low, high = run(basis, horizon, lookback, method)
    y = daily[basis]

    fig, ax = plt.subplots(figsize=FIGSIZE)
    _panel_sentiment_trend(ax, y, sent_class, sent_colors, OBJECT_NAME, start_date, end_date, interval=1)
    ax.set_title(f"Sentiment of {OBJECT_NAME}\n{y.index[0]:%d %b %Y} - {pred.index[-1]:%d %b %Y}",
                 fontsize=20, fontweight="bold", pad=20)

    line = pd.concat([y.iloc[[-1]], pred])   # mulai dari titik asli terakhir biar garisnya nyambung
    for s in sent_class:
        ax.plot(line.index, line[s], color=sent_colors[s], linewidth=3, linestyle="--", marker="o", ms=3)
        ax.fill_between(pred.index, low[s], high[s], color=sent_colors[s], alpha=0.12, linewidth=0)

    # angka di ujung garis, kalau tumpuk yang di atas digeser naik
    y_min, y_max = ax.get_ylim()
    gap, prev = 0.05 * (y_max - y_min), -np.inf
    for s in sorted(sent_class, key=lambda s: pred[s].iloc[-1]):
        prev = max(pred[s].iloc[-1], prev + gap)
        ax.annotate(f"{pred[s].iloc[-1]:,.0f} ({low[s].iloc[-1]:,.0f}–{high[s].iloc[-1]:,.0f})",
                    (pred.index[-1], prev), xytext=(8, 0), textcoords="offset points",
                    va="center", fontsize=12, color=sent_colors[s], fontweight="bold")
    ax.axvspan(y.index[-lookback], y.index[-1], color=COLOR_LEARNED, alpha=0.12)

    # keterangan grafik: warna sentimen + arti garis putus-putus, bayangan, kotak oranye
    handles, labels = ax.get_legend_handles_labels()
    handles += [Line2D([], [], color=COLOR_LEGEND, linewidth=3, linestyle="--"),
                Patch(color=COLOR_LEGEND, alpha=0.25), Patch(color=COLOR_LEARNED, alpha=0.25)]
    labels += ["Prediksi", f"Rentang wajar ({BAND:.0%} yakin)", f"Hari yang dipelajari ({lookback} hari)"]
    ax.legend(handles, labels, fontsize=12, loc="upper center", bbox_to_anchor=(0.5, 1.05), frameon=False, ncol=len(labels))

    dates = y.index.append(pred.index)
    ax.set_xticks(dates[::max(1, len(dates) // 15)])   # maksimal ~15 tanggal di sumbu bawah
    ax.set_xlabel("Tanggal", fontsize=18, fontweight="semibold")
    ax.set_ylabel(f"{basis} per hari", fontsize=18, fontweight="semibold")
    plt.tight_layout()
    plt.show()

    if method != AUTO and method != best:
        print(f'"{method}" nggak dipakai buat prediksi {horizon} hari, jadi diganti otomatis.')
    print(f"Metode: {best}" + (" (otomatis)" if method == AUTO else "") +
          f" | ketepatan {accuracy[best]:.0%} | diuji {n_tests} kali ke data lama")
    print("Semua metode: " + " · ".join(f"{m} {a:.0%}" for m, a in sorted(accuracy.items(), key=lambda x: -x[1])))
    if n_tests < MIN_TESTS:
        print(f"Data masih pendek (uji < {MIN_TESTS} kali), skor ketepatan masih kasar.")


out = w.interactive_output(draw, {"basis": basis_w, "horizon": horizon_w, "lookback": lookback_w, "method": method_w})
display(w.VBox([w.HBox([basis_w, method_w]), w.HBox([horizon_w, lookback_w]), out]))